# Suite JSN — JSON

JSON is a thin, strict (RFC 8259) encoding of Plain data, written exactly as Python's `json.dumps` writes it. These
cases cover strictness on output and input, text forms and encodings, value fidelity, and agreement with Plain.

In [ ]:
import { JSON, Plain, Proxies, Schemas as S } from "../src/Framework/index.js";
import { ValueError } from "../src/Framework/Errors.js";
import type { PlainMap } from "../src/Framework/Plain.js";
import { assert, equal, map, raises, same_graph, text } from "./support.js";

let Item = new S.OfObject.Builder().properties(text("s"), text("i", BigInt), text("f", Number), text("b", Boolean),
  text("raw", Uint8Array)).create();
const Link = new S.OfRelation.Builder().links("from", "to").properties(text("w", Number)).create();
Item = new S.OfObject.Builder(Item).relations((r) => r.name("out").of(Link).me("from"), (r) => r.name("in").of(Link).me("to")).update();
Proxies.register("Item", Item);
Proxies.register("Link", Link);
const B = Proxies.Builders;
const FromJSON = JSON.FromJSON(B);
const field = (textOut: string, name: string) =>
  (((JSON.loads(textOut) as PlainMap).get("objects") as PlainMap).get("s0") as PlainMap).get(name);

## JSN-01 · Output never contains NaN or Infinity literals

In [ ]:
for (const value of [NaN, Infinity, -Infinity]) {
  const out = JSON.ToJSON(Item, B.Item().f(value).create());
  assert(!/\b(NaN|Infinity)\b(?!")/.test(out.replaceAll('"NaN"', "").replaceAll('"Infinity"', "").replaceAll('"-Infinity"', "")));
  assert(["NaN", "Infinity", "-Infinity"].includes(field(out, "f") as string));
}
raises(ValueError, () => JSON.dumps(map({ x: NaN }))); // plain data should never hold one, and dumps refuses it

## JSN-02 · Input: non-standard constants, duplicate keys and syntax errors are rejected

In [ ]:
for (const bad of ['{"a": NaN}', '{"a": Infinity}', '{"a": -Infinity}', "[NaN]",
  '{"a": 1, "a": 2}', '{"x": {"a": 1, "a": 1}}', '[{"k": 1, "k": 1}]',
  '{"a": 1,}', "{'a': 1}", '{"a": 1} // c', '{"a": 01}', "", "   ", '{"a": 1} {"b": 2}', '{"a": .5}']) {
  raises(ValueError, () => JSON.loads(bad));
}

## JSN-03 · Text forms: str, UTF-8/16/32 bytes, and byte-order marks

In [ ]:
function encode(value: string, encoding: string): Uint8Array {
  const units = [...value].flatMap((c) => {
    const p = c.codePointAt(0) as number;
    return p > 0xffff ? [0xd800 + ((p - 0x10000) >> 10), 0xdc00 + ((p - 0x10000) & 0x3ff)] : [p];
  });
  const points = [...value].map((c) => c.codePointAt(0) as number);
  const u16 = (le: boolean) => units.flatMap((u) => (le ? [u & 0xff, u >> 8] : [u >> 8, u & 0xff]));
  const u32 = (le: boolean) => points.flatMap((p) => { const b = [p >>> 24, (p >> 16) & 0xff, (p >> 8) & 0xff, p & 0xff]; return le ? b.reverse() : b; });
  switch (encoding) {
    case "utf-16": return new Uint8Array([0xff, 0xfe, ...u16(true)]);
    case "utf-16-le": return new Uint8Array(u16(true));
    case "utf-16-be": return new Uint8Array(u16(false));
    case "utf-32": return new Uint8Array([0xff, 0xfe, 0x00, 0x00, ...u32(true)]);
    case "utf-8-sig": return new Uint8Array([0xef, 0xbb, 0xbf, ...new TextEncoder().encode(value)]);
    default: return new TextEncoder().encode(value);
  }
}
{
  const doc = '{"a": "é"}';
  assert(equal(JSON.loads(doc), map({ a: "é" })) && equal(JSON.loads(encode(doc, "utf-8")), map({ a: "é" })));
  for (const encoding of ["utf-16", "utf-16-le", "utf-16-be", "utf-32", "utf-8-sig"]) {
    assert(equal(JSON.loads(encode(doc, encoding)), map({ a: "é" })), encoding);
  }
  raises(ValueError, () => JSON.loads("﻿" + doc), "BOM"); // a BOM inside a str is an error; in bytes it is accepted
  raises(ValueError, () => JSON.loads(new Uint8Array([0xff, 0xfe, 0x00]))); // undecodable bytes
}

## JSN-04 · Output format: key order kept, non-ASCII kept, indentation optional

In [ ]:
{
  const item = B.Item().raw(new Uint8Array([1])).s("日本").i(1n).create();
  const compact = JSON.ToJSON(Item, item);
  assert(compact === '{"root": "s0", "objects": {"s0": {"s": "日本", "i": 1, "raw": "AQ=="}}}', compact);
  const pretty = JSON.ToJSON(Item, item, { indent: 2 });
  assert(pretty.includes("\n") && equal(JSON.loads(pretty), JSON.loads(compact)));
  assert(JSON.ToJSON.OfObject(Item, item, { indent: 4 }) === JSON.dumps(Plain.ToPlain(Item, item), { indent: 4 }));
}

## JSN-05 · Value fidelity: integers of any size, exact floats, every string

In [ ]:
for (const value of [0n, -1n, 2n ** 53n + 1n, 2n ** 100n, -(2n ** 200n)]) {
  assert((FromJSON(Item, JSON.ToJSON(Item, B.Item().i(value).create())) as any).i === value);
}
for (const value of [0.1, 1 / 3, 1e308, 5e-324, 2.2250738585072014e-308, -0.0, 123456789.123456789]) {
  const back = (FromJSON(Item, JSON.ToJSON(Item, B.Item().f(value).create())) as any).f;
  assert(Object.is(back, value));
}
for (const value of ["", "\x00", "\x1f\x7f", '"\\/', "  ", "\ud800", "\u{1f600}", "a\r\nb"]) {
  assert((FromJSON(Item, JSON.ToJSON(Item, B.Item().s(value).create())) as any).s === value, value);
}
{
  const all = new Uint8Array(Array.from({ length: 256 }, (_, i) => i));
  assert(equal((FromJSON(Item, JSON.ToJSON(Item, B.Item().raw(all).create())) as any).raw, all));
}

## JSN-06 · A lone surrogate is valid JSON text in memory but not encodable as UTF-8 (pinned)
Python raises when encoding it; JavaScript's `TextEncoder` silently replaces it with U+FFFD. Either way, callers
writing files should reject such strings.

In [ ]:
{
  const lone = JSON.ToJSON(Item, B.Item().s("\ud800").create());
  assert(lone.includes("\ud800"));
  const encoded = new TextDecoder().decode(new TextEncoder().encode(lone));
  assert(!encoded.includes("\ud800") && encoded.includes("�")); // lossy, where Python raises UnicodeEncodeError
}

## JSN-07 · Natives at the top level

In [ ]:
assert(JSON.ToJSON(new S.OfNative.Data(BigInt), 5n) === "5");
assert(JSON.ToJSON(new S.OfNative.Data(Uint8Array), new Uint8Array([0xff])) === '"/w=="');
assert(JSON.ToJSON.OfNative(new S.OfNative.Data(Number), Infinity) === '"Infinity"');
assert(FromJSON(new S.OfNative.Data(String), '"x"') === "x" && FromJSON.OfNative(new S.OfNative.Data(String), '"x"') === "x");
assert(JSON.ToJSON(new S.OfNative.Data(Number), 1.0) === "1.0"); // a float stays visibly a float
raises(TypeError, () => FromJSON(new S.OfNative.Data(BigInt), "true"));
raises(TypeError, () => FromJSON(new S.OfNative.Data(Number), "1")); // JSON 1 is an integer, not a float

## JSN-08 · JSON agrees with Plain, and graphs round trip

In [ ]:
{
  const hub = B.Item().s("hub").out((x: any) => x.to((y: any) => y.s("leaf1")).w(1.0))
    .out((x: any) => x.to((y: any) => y.s("leaf2")).w(NaN)).create();
  const graph = Plain.ToPlain.Reachable(Item, hub);
  const text_ = JSON.ToJSON.Reachable(Item, hub);
  assert(equal(JSON.loads(text_), graph));
  const back = FromJSON.Reachable(Item, text_) as Proxies.Instance;
  assert(back !== hub && same_graph(Plain.ToPlain.Reachable(Item, back), graph));
  raises(ValueError, () => FromJSON(Item, JSON.ToJSON(Item, hub)), "unresolved reference");
  raises(ValueError, () => FromJSON.Reachable(Item, text_.replace('"s1"', '"s9"')));
}

## JSN-09 · Parse errors match Python's `json` messages exactly
The table was generated from Python's `json` and is shared verbatim with the Python suite; this parser is its own.
Rows: input (text, or bytes as a list), expected message (or null when it parses), whether the message must match
exactly (codec errors from UTF-8/UTF-16 decoding match by prefix), and Python's repr of the parsed value.

In [ ]:
import { repr } from "../src/Framework/Repr.js";
{
  const TABLE = globalThis.JSON.parse(String.raw`[["", "Expecting value: line 1 column 1 (char 0)", true, null], ["   ", "Expecting value: line 1 column 4 (char 3)", true, null], ["-", "Expecting value: line 1 column 1 (char 0)", true, null], ["nul", "Expecting value: line 1 column 1 (char 0)", true, null], ["tru", "Expecting value: line 1 column 1 (char 0)", true, null], [".5", "Expecting value: line 1 column 1 (char 0)", true, null], ["[1,]", "Illegal trailing comma before end of array: line 1 column 3 (char 2)", true, null], ["{\"a\": 1,}", "Illegal trailing comma before end of object: line 1 column 8 (char 7)", true, null], ["{\"a\" 1}", "Expecting ':' delimiter: line 1 column 6 (char 5)", true, null], ["{1: 2}", "Expecting property name enclosed in double quotes: line 1 column 2 (char 1)", true, null], ["{\"a\": 1 \"b\": 2}", "Expecting ',' delimiter: line 1 column 9 (char 8)", true, null], ["[1 2]", "Expecting ',' delimiter: line 1 column 4 (char 3)", true, null], ["\"abc", "Unterminated string starting at: line 1 column 1 (char 0)", true, null], ["\"a\\", "Unterminated string starting at: line 1 column 1 (char 0)", true, null], ["\"a\\x\"", "Invalid \\escape: line 1 column 3 (char 2)", true, null], ["\"\\u12\"", "Invalid \\uXXXX escape: line 1 column 3 (char 2)", true, null], ["\"\\uZZZZ\"", "Invalid \\uXXXX escape: line 1 column 3 (char 2)", true, null], ["\"a\nb\"", "Invalid control character at: line 1 column 3 (char 2)", true, null], ["\"a\tb\"", "Invalid control character at: line 1 column 3 (char 2)", true, null], ["1 2", "Extra data: line 1 column 3 (char 2)", true, null], ["{\"a\": 1} x", "Extra data: line 1 column 10 (char 9)", true, null], ["[", "Expecting value: line 1 column 2 (char 1)", true, null], ["{", "Expecting property name enclosed in double quotes: line 1 column 2 (char 1)", true, null], ["{\"a\"", "Expecting ':' delimiter: line 1 column 5 (char 4)", true, null], ["{\"a\":", "Expecting value: line 1 column 6 (char 5)", true, null], ["01", "Extra data: line 1 column 2 (char 1)", true, null], ["1.", "Extra data: line 1 column 2 (char 1)", true, null], ["1e", "Extra data: line 1 column 2 (char 1)", true, null], ["[\n  1,\n  x]", "Expecting value: line 3 column 3 (char 9)", true, null], ["{\n\"a\": [\n1,\n2\n]\n,}", "Illegal trailing comma before end of object: line 6 column 1 (char 16)", true, null], ["NaN", "NaN is not valid JSON", true, null], ["-Infinity", "-Infinity is not valid JSON", true, null], ["[Infinity]", "Infinity is not valid JSON", true, null], ["99999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999", "Exceeds the limit (4300 digits) for integer string conversion: value has 4301 digits", true, null], [[0, 0, 0, 49], null, true, "1"], [[49, 0, 0, 0], null, true, "1"], [[254, 255, 0, 49], null, true, "1"], [[0, 49], null, true, "1"], [[49, 0], null, true, "1"], [[0, 0, 254, 255, 0, 0, 0, 49], null, true, "1"], [[0, 0, 0], "Expecting value: line 1 column 1 (char 0)", true, null], [[255, 254, 49], "'utf-16-le' codec can't decode", false, null], [[195, 40], "'utf-8' codec can't decode", false, null], [[255, 254, 0, 0, 0, 0, 17, 0], "'utf-32-le' codec can't decode bytes in position 4-7: code point not in range(0x110000)", true, null], [[0, 0, 0, 49, 50], "'utf-32-be' codec can't decode byte 0x32 in position 4: truncated data", true, null], [[0, 0, 0, 49, 50, 51], "'utf-32-be' codec can't decode bytes in position 4-5: truncated data", true, null], [[239, 187, 191, 49], null, true, "1"]]`) as [string | number[], string | null, boolean, string | null][];
  assert(TABLE.length === 47);
  for (const [source, message, exact, result] of TABLE) {
    const data = Array.isArray(source) ? new Uint8Array(source) : source;
    let value: unknown;
    try {
      value = JSON.loads(data);
    } catch (error) {
      if (!(error instanceof ValueError)) throw error;
      const text = (error as Error).message;
      assert(message !== null, `${String(source)} -> ${text}`);
      assert(exact ? text === message : text.startsWith(message), `${String(source)}: ${text}`);
      continue;
    }
    assert(message === null && repr(value) === result, `${String(source)} parsed as ${repr(value)}`);
  }
  assert(equal(JSON.loads(String.raw`["\ud83d\ude00", "\ud800", "\ud800x", "\udc00", "\b\f\n\r\t\/\\\""]`),
    ["\u{1f600}", "\ud800", "\ud800x", "\udc00", "\b\f\n\r\t/\\\""]));
}

## JSN-10 · Only plain data can be written
Python's `json` would turn an int, float, bool or None key into a string; plain data only has string keys.

In [ ]:
raises(TypeError, () => JSON.dumps(new Map([[1n, 2n]]) as never), "keys must be str, not int");
raises(TypeError, () => JSON.dumps(map({ a: [{ b: new Map([[1n, 2n]]) }] })), "keys must be str, not int");
for (const [value, name] of [[new Set([1n, 2n]), "set"], [new Uint8Array([120]), "bytes"], [{}, "object"]] as const) {
  raises(TypeError, () => JSON.dumps(map({ a: [] }).set("a", [value] as never)), `Object of type ${name} is not JSON serializable`);
}

## JSN-11 · Every kind of plain value is written and read, in Python's exact text

In [ ]:
{
  const value = [null, true, false, 0n, -1n, 1.5, "s", [], new Map(), map({ a: [null, {}] }), [[]]];
  assert(JSON.dumps(value) === '[null, true, false, 0, -1, 1.5, "s", [], {}, {"a": [null, {}]}, [[]]]');
  assert(JSON.dumps(value, { indent: 1 }) === '[\n null,\n true,\n false,\n 0,\n -1,\n 1.5,\n "s",\n [],\n {},\n {\n  "a": [\n   null,\n   {}\n  ]\n },\n [\n  []\n ]\n]');
  assert(equal(JSON.loads(JSON.dumps(value)), value) && equal(JSON.loads(JSON.dumps(value, { indent: 1 })), value));
  assert(JSON.loads(" null ") === null && equal(JSON.loads("{}"), new Map()) && equal(JSON.loads("[ ]"), []) && equal(JSON.loads("{ }"), new Map()));
}